# 任務 3：資料清理與處理決策
目標：比較刪除與填補，保留原始資料並檢查處理影響。

**學生版**｜與教師版任務及小節編號一一對應

## 開始之前
姓名：蔡雅雯＿＿＿＿　學號：C114118212＿＿＿＿

本教材可以在 Colab 或 Jupyter 執行，請依序執行儲存格。每份任務的準備儲存格會在記憶體重新建立相同的模擬資料，不會覆寫 CSV。若要分析自己先前產生的 CSV，先上傳檔案，執行準備儲存格後再執行 `df = pd.read_csv("student_learning_data.csv")`；不同資料的驗證數字可能不同。

成績是人為公式生成，資料品質問題在計分後加入。結果只描述模擬資料，不能推論真實學生或證明因果。每個任務請先預測，執行後修改與驗證，最後用自己的話解釋。

### 學生操作方式
1. 先執行下一個完整的資料準備儲存格。每份任務可獨立執行；重啟環境後請由上而下重新執行。
2. 在每個小節的程式中，將 `___` 替換成符合題意的程式碼，並把 `# 學生註解` 改成自己的繁體中文解說（至少說明指定程式的輸入／處理／輸出）。`___` 是刻意留下的空格，完成前執行會報錯。
3. 先記錄預測，執行後完成驗證與解釋；再完成延伸練習。遇到困難可以使用小節專屬 AI 提示詞，仍須檢查、修改、執行與理解。
4. 在文字儲存格中填寫答案，保留程式、執行結果、圖表和 AI 使用紀錄。依教師指示交付已完成的 `.ipynb`。首次執行任務 1 的存檔程式會覆寫同名 practice_student_data.csv，請先確認或改檔名。


In [1]:
# 載入分析與繪圖套件；np、pd、plt 是慣用縮寫
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 固定種子及呼叫順序，使本課範例能重現
rng = np.random.default_rng(42)
n = 60  # 模擬學生人數
departments = ["資訊管理", "企業管理", "金融管理"]
# 字典每個鍵是欄位名稱，每個值提供 60 位學生的資料
data = {
    "student_id": range(1, n + 1),
    "department": rng.choice(departments, n),
    "sleep_hours": rng.normal(6.7, 1.1, n).round(1),
    "screen_time_hours": rng.normal(5.0, 1.5, n).clip(1, 10).round(1),
    "study_hours": rng.normal(2.5, 1.0, n).clip(0.5, 6).round(1),
    "attendance_rate": rng.normal(88, 8, n).clip(60, 100).round(0),
}
df = pd.DataFrame(data)  # 把字典轉成資料表
noise = rng.normal(0, 5, n)  # 模擬其他未觀察的差異
score = (50 + 3 * df["study_hours"] + 2 * df["sleep_hours"]
         - 1.2 * df["screen_time_hours"] + 0.15 * df["attendance_rate"] + noise)
df["quiz_score"] = score.clip(0, 100).round(0)  # 限制 0 到 100 分
df.loc[[5, 17], "sleep_hours"] = np.nan  # 成績算完後才加入缺失
df.loc[8, "screen_time_hours"] = 15.0  # 成績算完後才加入異常值
print("資料已準備：", df.shape)

資料已準備： (60, 7)


## 3.1 保留原始資料
`copy()` 建立獨立副本；處理前保留原始資料，才能追查與比較。

### 學生練習 3.1：完成程式與註解

- **程式填空：** 補上建立獨立資料副本的指令。
- **註解任務：** 將程式中的兩處「學生註解」改寫為自己的說明；第一處解釋填空程式的輸入、處理與輸出，第二處解釋另一行程式的作用。
- **先預測：** 執行前，寫下你預期會看到什麼，或描述操作的效果。

**我的預測：** 我預測 copy() 會建立一份獨立的副本 df_clean，內容和 df 完全相同，大小都是 (60, 7)，兩邊的 sleep_hours 缺失數也都應該是 2 筆。之後如果只修改其中一邊，另一邊應該不會跟著改變

**AI 求助提示詞（可直接複製）：**
> 我是 Python 初學者，正在完成資料分析任務 3.1。我的資料表叫 df，請教我如何補上建立獨立資料副本的指令。 請先說明使用的 pandas 或 matplotlib 語法和每個參數，再給我可替換 `___` 的一小段程式；不要假設我的執行結果。請提供一個核對方法，並提醒我如何用自己的話撰寫註解。


In [3]:
# 學生註解 1：請說明填空程式的輸入、處理與輸出
# 並用 isna().sum() 數出 sleep_hours 的缺失筆數；
# 輸出＝df_clean 與兩邊缺失數的印出結果。
df_clean = df.copy()
print("原始缺失數：", df["sleep_hours"].isna().sum())
print("副本缺失數：", df_clean["sleep_hours"].isna().sum())
# 學生註解 2：請選另一行程式說明其作用
# copy() 會建立完全獨立的新資料表，
# 之後修改 df_clean 不會影響原本的 df，才能安全地做清洗與比較。

原始缺失數： 2
副本缺失數： 2


### 驗證、解釋與延伸 3.1

**驗證與解釋：** 原始與副本缺失數各是多少？為何保留原始資料？

**我的結果與解釋（引用實際輸出）：** 實際輸出為「原始缺失數： 2」、「副本缺失數： 2」，原始 df 和副本 df_clean 的 sleep_hours 缺失數都是 2 筆，代表 copy() 完整複製了資料，連缺失值都一併帶過來，兩邊內容一致。
為何要保留原始資料：原始 df 是最初準備好的乾淨基準，之後要清洗資料（補缺失、刪異常值）時，如果直接在 df 上修改，原始資料就回不來了，也沒法比較「清洗前」和「清洗後」的差異。先用 copy() 建立 df_clean，就能在副本上做修改，原始 df 保持不變，隨時可以對照兩者的列數、缺失數與數值差異。

**延伸練習：** 比較 df 和 df_clean 的列數與欄數。 請在下一個程式儲存格完成，並加上自己的註解。

**延伸練習的預測與驗證：** 我預測 df 和 df_clean 的列數、欄數會完全相同，都是 (60, 7)。實際執行 df.shape 和 df_clean.shape，兩者都輸出 (60, 7)，與預測一致，確認 copy() 複製出的副本與原始資料結構相同，之後可以各自修改而不互相影響。

**AI 求助提示詞：**
> 請引導我完成這個延伸練習：比較 df 和 df_clean 的列數與欄數。 目前資料表名稱為 df。請分步解釋程式，建議我如何檢查輸出，不要捏造結果；如果需要使用前面建立的變數，請指出其前置步驟。


In [4]:
# 在此撰寫延伸練習與自己的註解
# shape 回傳 (列數, 欄數)；兩邊應都是 (60, 7)，代表副本與原始資料結構相同。
print("df 的大小：", df.shape)
print("df_clean 的大小：", df_clean.shape)

df 的大小： (60, 7)
df_clean 的大小： (60, 7)


## 3.2 排除與填補
針對特定欄位排除缺失，並不是刪除原始資料。中位數填補保留筆數，但估計值會改變分布，也不會還原原始睡眠值。
**預測：** 兩種方案各保留幾筆？

### 學生練習 3.2：完成程式與註解

- **程式填空：** 補上使用中位數填補副本 sleep_hours 的運算。
- **註解任務：** 將程式中的兩處「學生註解」改寫為自己的說明；第一處解釋填空程式的輸入、處理與輸出，第二處解釋另一行程式的作用。
- **先預測：** 執行前，寫下你預期會看到什麼，或描述操作的效果。

**我的預測：** 兩種方案保留的筆數會不一樣：排除缺失法會直接把有缺值的列刪掉，剩下 58 筆；中位數填補法不會刪列，仍然是 60 筆，缺失數變成 0。填補後 sleep_hours 的平均值應該會稍微改變，但不會變動太多，因為只補了 2 筆、而且補的是中間值。

**AI 求助提示詞（可直接複製）：**
> 我是 Python 初學者，正在完成資料分析任務 3.2。我的資料表叫 df，請教我如何補上使用中位數填補副本 sleep_hours 的運算。 請先說明使用的 pandas 或 matplotlib 語法和每個參數，再給我可替換 `___` 的一小段程式；不要假設我的執行結果。請提供一個核對方法，並提醒我如何用自己的話撰寫註解。


In [5]:
# 學生註解 1：請說明填空程式的輸入、處理與輸出
# 輸入＝df_clean；處理＝dropna 先排出 sleep_hours 沒有缺值的子集合，
# 再取 median() 算出中位數，用 fillna() 把缺值補成這個中位數；
# 輸出＝sleep_complete（58 列）、df_clean（60 列、缺值已補滿）與幾組核對數字。
sleep_complete = df.dropna(subset=["sleep_hours"])
sleep_median = df_clean["sleep_hours"].median()
df_clean["sleep_hours"] = df_clean["sleep_hours"].fillna(sleep_median)
print("原始筆數：", len(df), "排除後：", len(sleep_complete))
print("填補中位數：", sleep_median)
print("原始與填補後平均：", df["sleep_hours"].mean(), df_clean["sleep_hours"].mean())
print("原始缺失數：", df["sleep_hours"].isna().sum())
# 學生註解 2：請選另一行程式說明其作用
# len() 是在數資料有幾列，用它比較原始 60 列和排除後剩下幾列，
# 可以看出 dropna 到底刪掉幾筆資料。

原始筆數： 60 排除後： 58
填補中位數： 6.75
原始與填補後平均： 6.682758620689653 6.684999999999999
原始缺失數： 2


### 驗證、解釋與延伸 3.2

**驗證與解釋：** 排除與填補各保留多少列？原始及副本缺失數與平均數如何變化？

**我的結果與解釋（引用實際輸出）：** 實際輸出為「原始筆數： 60 排除後： 58」、「填補中位數： 6.75」、「原始缺失數： 2」，填補後再用 isna().sum() 檢查則是 0。

排除法把有缺值的 2 列整列刪除，剩下 58 列，平均是 6.6828；填補法保留 60 列，用中位數 6.75 補那 2 筆，平均從原本的 6.6828 變成 6.685，只差約 0.002。所以原始與副本的差異是：df 仍有 2 筆缺值、df_clean 是 0 筆缺值但其中 2 筆是補出來的估計值。
用中位數是因為它不受極端值影響，補進去較不會把整體分布拉偏；缺點是那 2 筆並非學生真實睡眠時數，後續解讀時要記得它們是估計值。

**延伸練習：** 另建副本，以平均數填補睡眠缺失；比較中位數方案的平均值與缺失數。 請在下一個程式儲存格完成，並加上自己的註解。

**延伸練習的預測與驗證：** 我預測改用平均數填補，缺失數一樣會變成 0、一樣保留 60 列，但填補後的平均值會跟排除法的平均幾乎相同，因為補進去的就是平均值本身。實際執行後，平均數填補法補入 6.6828，填補後平均是 6.6828；中位數方案是 6.685，兩者只差 0.002 左右，缺失數都是 0、都保留 60 列，與預測一致。由此可以看出缺值只有 2 筆時，兩種填補方式對平均值的影響都很小。

**AI 求助提示詞：**
> 請引導我完成這個延伸練習：另建副本，以平均數填補睡眠缺失；比較中位數方案的平均值與缺失數。 目前資料表名稱為 df。請分步解釋程式，建議我如何檢查輸出，不要捏造結果；如果需要使用前面建立的變數，請指出其前置步驟。


In [6]:
# 在此撰寫延伸練習與自己的註解
# 另建副本，改用平均數填補睡眠缺值，再與中位數方案比較。
# 先從原始 df 複製，避免重複修改同一份資料。
df_mean = df.copy()
sleep_mean = df_mean["sleep_hours"].mean()
df_mean["sleep_hours"] = df_mean["sleep_hours"].fillna(sleep_mean)

print("平均數填補值：", sleep_mean, "填補後平均：", df_mean["sleep_hours"].mean())
print("平均數方案缺失數：", df_mean["sleep_hours"].isna().sum(), "筆數：", len(df_mean))
print("中位數方案平均：", df_clean["sleep_hours"].mean(), "筆數：", len(df_clean))

平均數填補值： 6.682758620689653 填補後平均： 6.682758620689655
平均數方案缺失數： 0 筆數： 60
中位數方案平均： 6.684999999999999 筆數： 60


## 3.3 處理異常
本案例無法從現有資料還原學生 9 在計分時的螢幕時間。可保留並註記，或另建篩選後資料作比較，避免憑空改值。

### 學生練習 3.3：完成程式與註解

- **程式填空：** 補上利用 within_range 建立篩選後副本。
- **註解任務：** 將程式中的兩處「學生註解」改寫為自己的說明；第一處解釋填空程式的輸入、處理與輸出，第二處解釋另一行程式的作用。
- **先預測：** 執行前，寫下你預期會看到什麼，或描述操作的效果。

**我的預測：** 我預測 screen_time_hours 大多是 1～10 小時以內，只有 1 筆異常值（15 小時）會被排除，所以 df_screen_filtered 應該保留 59 筆，被排除的學生是第 9 號。

**AI 求助提示詞（可直接複製）：**
> 我是 Python 初學者，正在完成資料分析任務 3.3。我的資料表叫 df，請教我如何補上利用 within_range 建立篩選後副本。 請先說明使用的 pandas 或 matplotlib 語法和每個參數，再給我可替換 `___` 的一小段程式；不要假設我的執行結果。請提供一個核對方法，並提醒我如何用自己的話撰寫註解。


In [7]:
# 學生註解 1：請說明填空程式的輸入、處理與輸出
# 輸入＝df；處理＝用布林條件「screen_time_hours ≤ 10」挑出合理範圍的列，
# 建立篩選後的副本 df_screen_filtered；
# 輸出＝保留筆數、以及被排除學生的 student_id 清單。
within_range = df["screen_time_hours"] <= 10
df_screen_filtered =  df[within_range].copy()
print("保留筆數：", len(df_screen_filtered))
print("被排除的學生：", df.loc[~within_range, "student_id"].tolist())
# 學生註解 2：請選另一行程式說明其作用
# ~within_range 是把條件反過來，變成「不滿足條件」的列，
# 用來找出被排除的是哪些學生，核對到底刪掉了誰。


保留筆數： 59
被排除的學生： [9]


### 驗證、解釋與延伸 3.3

**驗證與解釋：** 保留幾筆、排除哪位？是否能由現有資料還原原始螢幕時間？

**我的結果與解釋（引用實際輸出）：** 實際輸出「保留筆數： 59」、「被排除的學生： [9]」。原始 60 筆，過濾後剩 59 筆，排除 1 筆，正是學號 9、screen_time_hours 為 15.0 的那一列。能否由現有資料還原被排除的螢幕時間？可以，因為 df_screen_filtered 只是「篩選後的檢視」，原始 df 完全沒動。只要用 df.loc[~within_range] 找出被排除的列，就能看到該學生完整的一整行資料，包括 15.0 小時的螢幕時間；所以被排除的值沒有真正遺失，只是被移出了分析範圍。

**延伸練習：** 核對原始筆數是否等於保留加排除筆數。 請在下一個程式儲存格完成，並加上自己的註解。

**延伸練習的預測與驗證：** 我預測原始筆數 60 應該等於保留筆數 59 加排除筆數 1。實際用 len(df) == len(df_screen_filtered) + int((~within_range).sum()) 核對，結果是 60 = 59 + 1，等式成立，確認沒有漏刪或重複刪除。另外檢查 df_screen_filtered 的 screen_time_hours 最大值是 9.4，已沒有超過 10 的異常值，代表篩選有效。

**AI 求助提示詞：**
> 請引導我完成這個延伸練習：核對原始筆數是否等於保留加排除筆數。 目前資料表名稱為 df。請分步解釋程式，建議我如何檢查輸出，不要捏造結果；如果需要使用前面建立的變數，請指出其前置步驟。


In [8]:
# 在此撰寫延伸練習與自己的註解
# 核對原始筆數是否等於保留加排除筆數。
# within_range.sum() 是保留（True）的筆數，(~within_range).sum() 是排除（False）的筆數。
kept = int(within_range.sum())
removed = int((~within_range).sum())
print("原始：", len(df), "保留：", kept, "排除：", removed)
print("保留 + 排除 =", kept + removed, "是否等於原始：", kept + removed == len(df))

# 再確認過濾後已無超過 10 的螢幕時間
print("過濾後最大螢幕時間：", df_screen_filtered["screen_time_hours"].max())

原始： 60 保留： 59 排除： 1
保留 + 排除 = 60 是否等於原始： True
過濾後最大螢幕時間： 9.4


## 任務交付與 AI 使用紀錄

完成本任務的三個小節，儲存含輸出的 Notebook，依教師指定的方式繳交。任務 8 另需完成 8.3 的完整分析報告及其佐證圖表。

- 使用的 AI 提示詞（未使用請填「未使用」）：我是 Python 初學者，正在完成資料分析任務 3.3，我的資料表叫 df，請教我如何補上利用 within_range 建立篩選後副本的指令。請先說明使用的 pandas 語法和每個參數，再給我可替換空格的一小段程式；不要假設我的執行結果，請提供一個核對方法，並提醒我如何用自己的話撰寫註解。（任務 3 的 3.1、3.2 亦使用相同格式，分別改為建立 copy（） 副本、以及用中位數填補 sleep_hours 缺值。）
- AI 建議中我修改或拒絕的一處及原因：AI 建議篩選後要加上 copy() 才不會出現 SettingWithCopyWarning。我採用了，因為只寫「df_screen_filtered = df[within_range]」得到的是參照而非獨立副本，之後若在這個副本上改值，可能會連動影響原始 df，也無法確定修改是否成功；加上 copy() 後它就是獨立資料表，原始 df 保持不變，才能安心比較前後差異。
- 遇到的錯誤或疑問、修正與驗證方式：排除缺失與中位數填補到底差多少，以及被排除的異常值會不會找不回來。做法是分別印出 len(df)、len(sleep_complete)、len(df_clean)、缺失數與平均值來核對，發現排除法剩 58 筆、平均 6.6828，填補法仍是 60 筆、平均從 6.6828 變成 6.685，差異很小；另外用「條件取反」的方式列出被排除的學生，確認是學號 9、螢幕時間 15.0，資料沒有真的消失，因此修正了我原本以為篩選會遺失資料的想法。
- 本任務一項有數據支持的發現及對應儲存格：sleep_hours 只有 2 筆缺值，所以用中位數 6.75 填補前後，這一欄的平均值只從 6.6828 變成 6.685，變動約 0.002，代表缺值數量極少時，填補與排除兩種方案對統計結果影響都很小。對應 3.2 的儲存格（dropna、median、fillna 與平均值比較那格）。
- 一項資料或推論限制：填補進去的 2 筆睡眠時數是中位數估計值，不是學生真實回報的資料，且學號 9 的螢幕時間 15 小時只是因超過 10 小時的假設門檻被排除，資料本身並未說明它是輸入錯誤還是該生真的使用時間很長，所以不能據此宣稱這筆是錯值。

**交付前檢核：** 已填完程式空格與註解；已填寫預測、驗證、解釋與延伸；已由上到下執行並確認圖表和數字與文字一致。
